In [88]:
import sys
print(sys.executable)

e:\LangChain\langchain_learning\gpu_env\Scripts\python.exe


In [89]:
import torch

print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA build: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [90]:
import os

print(os.environ.get("HF_HOME"))
print(os.environ.get("HUGGINGFACE_HUB_CACHE"))

E:/huggingface
E:\huggingface\hub


In [91]:
!pip install -U langgraph langchain langchain-huggingface huggingface_hub langchain-community langchain_core requests duckduckgo-search ddgs transformers accelerate sentencepiece 


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [92]:
import os
import torch
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain.agents import create_agent  # Modern agent factory

In [93]:
# 1. Environment Configurations
os.environ["HF_HOME"] = "E:/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "E:/huggingface/models"

In [94]:
# 2. Define Tools
search_tool = DuckDuckGoSearchRun()

@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    """
    # Note: Corrected your API URL key string concatenation issue here
    url = f"https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}"
    response = requests.get(url)
    return str(response.json())

tools = [search_tool, get_weather]

In [97]:
# 3. Model Loading
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,          # Prevents RAM duplication
    offload_folder="E:/hf_offload"   # Offloads excess layers to disk instead of crashing RAM
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500, # Increased to allow room for reasoning + tool structures
)


OSError: The paging file is too small for this operation to complete. (os error 1455)

In [ ]:
# Base pipeline (doesn't support tools)
raw_llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# FIX 1: Wrap with ChatHuggingFace to add tool-binding capabilities (.bind_tools())
chat_model = ChatHuggingFace(llm=raw_llm)

In [ ]:
# 4. Create Modern Agent 
# FIX 2: Modern create_agent returns a fully functional runner graph.
# It automatically manages execution, so you do NOT use AgentExecutor.
agent = create_agent(
    model=chat_model, # Pass the wrapped tool-capable chat model
    tools=tools,
    system_prompt="You are a helpful assistant. Utilize your tools sequentially to solve user queries."
)

In [ ]:
# 5. Invoke 
# FIX 3: Pass inputs via the standard "messages" list structure required by modern agents.
response = agent.invoke({
    "messages": [
        ("user", "Find the capital of Madhya Pradesh, then find it's current weather condition")
    ]
})

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
# Print the final result message
print(response["messages"][-1].content)

<|im_start|>system
You are a helpful assistant. Utilize your tools sequentially to solve user queries.<|im_end|>
<|im_start|>user
Find the capital of Madhya Pradesh, then find it's current weather condition<|im_end|>
<|im_start|>assistant
The capital of Madhya Pradesh is Bhopal.

To find the current weather condition in Bhopal, I would need access to real-time weather data which is not available through this platform. However, you can easily check the current weather conditions for Bhopal using any online weather service or app. Just enter "Bhopal" into their search bar, and they will provide you with the latest updates on temperature, humidity, precipitation, and other relevant information.
